In [9]:
#导入必须的包
from langchain_community.document_loaders import UnstructuredExcelLoader, Docx2txtLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
#导入聊天所需的模块
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv

# 加载环境
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")


#定义chatdoc
# Define the ChatDoc class
class ChatDoc():

	def __init__(self):
		self.doc = None
		self.splitText = []  #分割后的文本 split text
		self.template = [
			("system",
			 "你是一个处理文档的秘书,你从不说自己是一个大模型或者AI助手,你会根据下面提供的上下文内容来继续回答问题.\n 上下文内容\n {context} \n"),
			("human", "你好！"),
			("ai", "您好，我是尚硅谷秘书"),
			("human", "{question}"),
		]
		self.prompt = ChatPromptTemplate.from_messages(self.template)

	def getFile(self):
		doc = self.doc

		loaders = {
			"docx": Docx2txtLoader
		}
		file_extension = doc.split(".")[-1]
		loader_class = loaders.get(file_extension)
		if loader_class:
			try:
				loader = loader_class(doc)
				text = loader.load()
				return text
			except Exception as e:
				print(f"Error loading {file_extension} files:{e}")
		else:
			print(f"Unsupported file extension: {file_extension}")
			return None

	#处理文档的函数
	def splitSentences(self):
		full_text = self.getFile()  #获取文档内容 get the content of the document
		if full_text != None:
			#对文档进行分割
			# Split the document
			text_split = CharacterTextSplitter(
				chunk_size=500,
				chunk_overlap=100,
				separator="\n\n",
				length_function=len,
				is_separator_regex=False
			)
		texts = text_split.split_documents(full_text)
		self.splitText = texts

	#向量化与向量存储
	def embeddingAndVectorDB(self):
		# embeddings = OpenAIEmbeddings(
		# model="BAAI/bge-m3",
		# api_key=os.getenv("SILICON_API_KEY"),
		# base_url="https://api.siliconflow.cn/v1"
		# )
		embeddings = OpenAIEmbeddings()
		db = Chroma.from_documents(
			documents=self.splitText,
			embedding=embeddings,
		)
		return db

	#提问并找到相关的文本块
	def askAndFindFiles(self, question):
		db = self.embeddingAndVectorDB()
		print(db._collection.count())
		#retriever = db.as_retriever(search_type="mmr")
		retriever = db.as_retriever()
		return retriever.invoke(input=question)

	# 用自然语言和文档聊天
	def chatWithDoc(self, question):
		_content = ""
		context = self.askAndFindFiles(question)
		for i in context:
			_content += i.page_content
		print(f"{_content}", "_content")
		messages = self.prompt.format_messages(context=_content, question=question)
		print("message:", messages)
		llm = ChatOpenAI(model="gpt-4o-mini",
		                 temperature=0,
		                 )
		return llm.invoke(messages)


chat_doc = ChatDoc()
chat_doc.doc = "./asset/load/13-sgg_chat.docx"
chat_doc.splitSentences()

while True:
	question = input("请输入你的问题，输入exit退出！")
	if question == "exit":
		break
	print(f"Q:{question}")
	response = chat_doc.chatWithDoc(question)
	print(f"A:{response.content}")

Q:你好
12
尚硅谷师资团队评估报告

一、机构基本信息

名称：尚硅谷教育科技有限公司
总部地址：北京市海淀区中关村软件园创新大厦B座5层
成立日期：2016年3月15日
法定代表人：张明哲
注册资本：人民币2000万元
员工规模：全职讲师150人，技术助教300人
官方电话：010-66669999
官方邮箱：hr@atguigu.com

二、师资团队核心优势

技术背景深厚

90%讲师来自BAT、字节跳动等一线互联网企业，平均技术从业年限8年+

35%讲师拥有开源项目贡献经历（如Apache、Linux基金会项目）

教学能力突出

所有讲师均通过“教学能力三级认证体系”（技术能力、课程设计、课堂表达）

学员平均评分4.9/5.0（2023年内部调研数据）

行业影响力显著

12位讲师担任Oracle/Red Hat/AWS认证官方考官

出版技术书籍28本，累计销量超50万册

三、师资结构分析

类别

人数

占比

典型代表

架构师级讲师

25

16.7%

前阿里P9分布式系统专家

高级研发讲师

80

53.3%

腾讯T4级后台开发工程师

新兴领域讲师尚硅谷师资团队评估报告

一、机构基本信息

名称：尚硅谷教育科技有限公司
总部地址：北京市海淀区中关村软件园创新大厦B座5层
成立日期：2016年3月15日
法定代表人：张明哲
注册资本：人民币2000万元
员工规模：全职讲师150人，技术助教300人
官方电话：010-66669999
官方邮箱：hr@atguigu.com

二、师资团队核心优势

技术背景深厚

90%讲师来自BAT、字节跳动等一线互联网企业，平均技术从业年限8年+

35%讲师拥有开源项目贡献经历（如Apache、Linux基金会项目）

教学能力突出

所有讲师均通过“教学能力三级认证体系”（技术能力、课程设计、课堂表达）

学员平均评分4.9/5.0（2023年内部调研数据）

行业影响力显著

12位讲师担任Oracle/Red Hat/AWS认证官方考官

出版技术书籍28本，累计销量超50万册

三、师资结构分析

类别

人数

占比

典型代表

架构师级讲师

25

16.7%

前阿里P9分布式系统专家

高级研发讲师

80

53.3%

腾讯T4级后台开发工程师

新兴领域